# Person 3 — Poolability Metrics, Emissions Savings & Scale-Up

**Goal:** Using cluster labels from Persons 1 & 2, define "poolable trips", measure pooling rates per cluster, and quantify CO₂ savings.

**Inputs from other notebooks:**
- `../Data/trips_dbscan_labeled.csv` (Person 1)
- `../Data/trips_kmeans_labeled.csv` (Person 2)
- `../Data/clustering_comparison.csv` (Person 2)

**A trip pair is "poolable" if:**
- Same cluster (same pickup zone at the same time)
- Pickup time gap ≤ `TIME_WINDOW_MIN` minutes
- Trips are not already a `shared_match_flag == 'Y'` pair

**Outputs:**
- % poolable trips per cluster, per algorithm
- Estimated CO₂ savings from pooling
- Final summary table comparing DBSCAN vs K-Means on poolability
- Scale-up analysis: how metrics change as dataset grows

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["figure.dpi"] = 120

# Poolability parameters — tune these
TIME_WINDOW_MIN = 5    # trips within 5 minutes of each other can be pooled
POOL_RATIO      = 0.5  # assume pooled trips split the per-mile emissions equally

## 1. Load Labeled Data from Persons 1 & 2

In [ ]:
db_df = pd.read_csv("../Data/trips_dbscan_labeled.csv")
km_df = pd.read_csv("../Data/trips_kmeans_labeled.csv")
comp  = pd.read_csv("../Data/clustering_comparison.csv")

for _df, name in [(db_df, "DBSCAN"), (km_df, "K-Means")]:
    _df["pickup_datetime"]  = pd.to_datetime(_df["pickup_datetime"],  errors="coerce")
    _df["dropoff_datetime"] = pd.to_datetime(_df["dropoff_datetime"], errors="coerce")
    _df["co2_g_per_mile"]   = pd.to_numeric(_df.get("co2_g_per_mile",  _df.get("co2TailpipeGpm", np.nan)), errors="coerce")
    _df["trip_miles"]       = pd.to_numeric(_df["trip_miles"], errors="coerce")
    print(f"{name}: {len(_df):,} rows, columns: {list(_df.columns[:8])} ...")

print("\nAlgorithm comparison:")
print(comp.to_string(index=False))

## 2. Baseline Emissions (No Pooling)

In [ ]:
def total_co2(df):
    """Total CO₂ (grams) assuming every trip is solo."""
    return (df["trip_miles"] * df["co2_g_per_mile"]).sum()

baseline_db = total_co2(db_df)
baseline_km = total_co2(km_df)
print(f"Baseline CO₂ (DBSCAN subset) : {baseline_db:,.0f} g  ({baseline_db/1e6:.2f} kg)")
print(f"Baseline CO₂ (K-Means subset): {baseline_km:,.0f} g  ({baseline_km/1e6:.2f} kg)")

## 3. Define & Count Poolable Trips

For each cluster, sort trips by `pickup_datetime` and count consecutive pairs with a time gap ≤ `TIME_WINDOW_MIN`.  
This is a lower-bound estimate — a full optimizer (covering the next project stage) would do better.

In [ ]:
def count_poolable(df, cluster_col, time_window_min=TIME_WINDOW_MIN):
    """
    Returns (n_poolable_trips, pct_poolable, per_cluster_df).
    Excludes noise cluster (-1 in DBSCAN).
    """
    tw = pd.Timedelta(minutes=time_window_min)
    poolable_indices = set()
    cluster_stats    = []

    valid = df[df[cluster_col] != -1].copy()

    for cluster_id, grp in valid.groupby(cluster_col):
        grp_sorted = grp.sort_values("pickup_datetime")
        times      = grp_sorted["pickup_datetime"].values
        idxs       = grp_sorted.index.tolist()

        pool_in_cluster = set()
        for i in range(len(times) - 1):
            gap = pd.Timestamp(times[i + 1]) - pd.Timestamp(times[i])
            if gap <= tw:
                pool_in_cluster.add(idxs[i])
                pool_in_cluster.add(idxs[i + 1])

        poolable_indices |= pool_in_cluster
        cluster_stats.append({
            "cluster_id":  cluster_id,
            "n_trips":     len(grp),
            "n_poolable":  len(pool_in_cluster),
            "pct_poolable": len(pool_in_cluster) / len(grp) * 100 if len(grp) > 0 else 0,
        })

    n_poolable  = len(poolable_indices)
    pct_poolable = n_poolable / len(df) * 100
    per_cluster  = pd.DataFrame(cluster_stats).sort_values("pct_poolable", ascending=False)
    return n_poolable, pct_poolable, per_cluster, poolable_indices

In [ ]:
n_pool_db,  pct_pool_db,  cluster_stats_db, pool_idx_db  = count_poolable(db_df, "dbscan_cluster")
n_pool_km,  pct_pool_km,  cluster_stats_km, pool_idx_km  = count_poolable(km_df, "kmeans_cluster")

print(f"DBSCAN  — Poolable trips: {n_pool_db:,}  ({pct_pool_db:.1f}% of subset)")
print(f"K-Means — Poolable trips: {n_pool_km:,}  ({pct_pool_km:.1f}% of subset)")

## 4. Estimate CO₂ Savings from Pooling

Assumption: pooled trips share one vehicle, so total vehicle-miles (and thus emissions) are halved for pooled pairs.

In [ ]:
def co2_with_pooling(df, pool_idx, pool_ratio=POOL_RATIO):
    """Recalculate total CO₂ assuming pooled trips share a vehicle."""
    co2_solo   = df.loc[~df.index.isin(pool_idx), "trip_miles"] * df.loc[~df.index.isin(pool_idx), "co2_g_per_mile"]
    co2_pooled = df.loc[df.index.isin(pool_idx),  "trip_miles"] * df.loc[df.index.isin(pool_idx),  "co2_g_per_mile"] * pool_ratio
    return co2_solo.sum() + co2_pooled.sum()

co2_after_db = co2_with_pooling(db_df, pool_idx_db)
co2_after_km = co2_with_pooling(km_df, pool_idx_km)

savings_db = (baseline_db - co2_after_db) / baseline_db * 100
savings_km = (baseline_km - co2_after_km) / baseline_km * 100

print(f"DBSCAN  — CO₂ after pooling: {co2_after_db/1e6:.2f} kg  (saves {savings_db:.1f}%)")
print(f"K-Means — CO₂ after pooling: {co2_after_km/1e6:.2f} kg  (saves {savings_km:.1f}%)")

## 5. Final Summary Table

In [ ]:
final_summary = pd.DataFrame([
    {
        "Algorithm":       "DBSCAN",
        "N Clusters":      comp.loc[comp["algorithm"]=="DBSCAN",  "n_clusters"].values[0],
        "Noise %":         comp.loc[comp["algorithm"]=="DBSCAN",  "noise_pct"].values[0],
        "Silhouette":      comp.loc[comp["algorithm"]=="DBSCAN",  "silhouette"].values[0],
        "Davies-Bouldin":  comp.loc[comp["algorithm"]=="DBSCAN",  "davies_bouldin"].values[0],
        "% Poolable":      round(pct_pool_db, 1),
        "CO₂ Savings %":   round(savings_db, 1),
    },
    {
        "Algorithm":       "K-Means",
        "N Clusters":      comp.loc[comp["algorithm"]=="K-Means", "n_clusters"].values[0],
        "Noise %":         0.0,
        "Silhouette":      comp.loc[comp["algorithm"]=="K-Means", "silhouette"].values[0],
        "Davies-Bouldin":  comp.loc[comp["algorithm"]=="K-Means", "davies_bouldin"].values[0],
        "% Poolable":      round(pct_pool_km, 1),
        "CO₂ Savings %":   round(savings_km, 1),
    },
])

print("=" * 80)
print(final_summary.to_string(index=False))
print("=" * 80)

final_summary.to_csv("../Data/final_clustering_summary.csv", index=False)
print("Saved → ../Data/final_clustering_summary.csv")

## 6. Visualization: Poolable vs Non-Poolable & CO₂ Savings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar: % poolable by algorithm
axes[0].bar(["DBSCAN", "K-Means"], [pct_pool_db, pct_pool_km], color=["steelblue", "coral"], width=0.4)
axes[0].set_ylabel("% of trips poolable")
axes[0].set_title(f"% Poolable Trips (time window = {TIME_WINDOW_MIN} min)")
axes[0].set_ylim(0, 100)
for i, v in enumerate([pct_pool_db, pct_pool_km]):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha="center", fontsize=11)

# Bar: CO₂ savings
axes[1].bar(["DBSCAN", "K-Means"], [savings_db, savings_km], color=["steelblue", "coral"], width=0.4)
axes[1].set_ylabel("CO₂ savings (%)")
axes[1].set_title("Estimated CO₂ Reduction from Pooling")
axes[1].set_ylim(0, max(savings_db, savings_km) * 1.25 + 1)
for i, v in enumerate([savings_db, savings_km]):
    axes[1].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=11)

plt.tight_layout()
plt.savefig("poolability_summary.png", bbox_inches="tight")
plt.show()

## 7. Scale-Up Analysis

Run the best-performing algorithm at progressively larger sample sizes and track whether metrics stay stable.

> **Coordinate with Person 1**: once DBSCAN epsilon is tuned on 10k, Person 1 exports scaled-up files; update `SCALE_FILES` below.

In [ ]:
# TODO — fill in once Person 1 has exported scaled-up CSVs
# SCALE_FILES = {
#     10_000:  "../Data/trips_dbscan_labeled.csv",
#     100_000: "../Data/trips_dbscan_labeled_100k.csv",
#     # "full": "../Data/trips_dbscan_labeled_full.csv",
# }

# scale_results = []
# for n, path in SCALE_FILES.items():
#     _df = pd.read_csv(path)
#     _df["pickup_datetime"] = pd.to_datetime(_df["pickup_datetime"], errors="coerce")
#     _df["co2_g_per_mile"]  = pd.to_numeric(_df.get("co2_g_per_mile", _df.get("co2TailpipeGpm", np.nan)), errors="coerce")
#     _df["trip_miles"]      = pd.to_numeric(_df["trip_miles"], errors="coerce")
#     n_pool, pct_pool, _, pool_idx = count_poolable(_df, "dbscan_cluster")
#     baseline = total_co2(_df)
#     after    = co2_with_pooling(_df, pool_idx)
#     scale_results.append({"n_rows": n, "pct_poolable": round(pct_pool, 2), "co2_savings_pct": round((baseline-after)/baseline*100, 2)})

# print(pd.DataFrame(scale_results).to_string(index=False))

print("Scale-up section ready — uncomment once Person 1 exports larger files.")